# 04-plan: the three-way benchmark

Greedy, CP-SAT on its own and CP-SAT from greedy, through `planning.plan()` over the lever ranges on the full-area tables (issue #30). Run on an idle machine; the load is printed.

## Setup

In [ ]:
import os
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
import ortools
import pandas as pd

from catchment import assumptions, planning
from catchment.fetch import DATA

FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
BAR_S = 10   # a plan returns within this many seconds
EPS = 1e-6   # tolerance when comparing orders a day

inputs = planning.load_inputs()
print(f"{len(inputs.hex_ids)} hexes, {len(inputs.site_ids)} candidate sites, {len(inputs.pair_site)} pairs")
print(f"cpu_count {os.cpu_count()}, ortools {ortools.__version__}")
print(f"PLAN_BUDGET {assumptions.PLAN_BUDGET} s, WORKERS {assumptions.WORKERS}, SOLVER_SEED {assumptions.SOLVER_SEED}")
print(f"load average at start (1, 5, 15 min): {os.getloadavg()}")

## Grid

In [ ]:
SWEEPS: dict[str, list[float]] = {
    "n_new": [1, 5, 15, 20],
    "promise": [8, 12, 14, 16],
    "multiplier": [1.0, 1.4, 2.2, 2.5],
    "capacity": [0.5, 0.75, 1.5, 2.0],
    "capture": [0.5, 0.75, 1.25, 1.5],
    "competitor_density": [0.0, 0.5, 1.5, 2.0],
}
default = planning.Scenario()
grid: list[tuple[str, str, planning.Scenario]] = [("default", "default", default)]
for lever, points in SWEEPS.items():
    low, high = planning.LEVER_RANGES[lever]
    for value in points:
        assert low <= value <= high, (lever, value)
        assert value != getattr(default, lever), (lever, value)   # the default is not repeated
        grid.append((lever, f"{lever}={value:g}", planning.Scenario(**{lever: value})))
grid.append(("corner", "n_new=20, capacity=0.5", planning.Scenario(n_new=20, capacity=0.5)))
grid.append(("corner", "capture=1.5, density=0", planning.Scenario(capture=1.5, competitor_density=0.0)))
for k in range(1, 5):
    grid.append(("seed", f"seed={assumptions.SEED + k}", planning.Scenario(seed=assumptions.SEED + k)))
print(len(grid), "scenario rows,", len(grid) * len(planning.METHODS), "plan() calls")

## Run

In [ ]:
records: list[dict[str, object]] = []
t_grid = time.perf_counter()
for group, label, sc in grid:
    for method in planning.METHODS:
        load_before = os.getloadavg()[0]
        p = planning.plan(inputs, sc, method)
        records.append({
            "group": group, "label": label,
            "n_new": sc.n_new, "promise": sc.promise, "multiplier": sc.multiplier, "capacity": sc.capacity,
            "capture": sc.capture, "competitor_density": sc.competitor_density, "seed": sc.seed,
            "method": method,
            "demand_covered": p.demand_covered, "households_covered": p.households_covered,
            "lost_to_capacity": p.lost_to_capacity, "unreached": p.unreached,
            "slow_traffic_coverage": p.slow_traffic_coverage,
            "slow_above_covered": p.slow_traffic_coverage > p.demand_covered + EPS,
            "status": p.status, "objective": p.objective, "bound": p.bound, "gap": p.gap,
            "build_seconds": p.build_seconds, "solve_seconds": p.solve_seconds, "seconds": p.seconds,
            "load_before": load_before})
        print(f"{label:28s} {method:18s} {p.status:9s} covered {p.demand_covered:9.1f} gap {p.gap} {p.seconds:6.2f} s load {load_before:.1f}")
grid_wall_s = time.perf_counter() - t_grid
runs = pd.DataFrame(records)
peak_rss_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9   # bytes on macOS
print(f"grid wall time {grid_wall_s / 60:.1f} min, peak RSS {peak_rss_gb:.2f} GB")

## Table

In [ ]:
KEYS = ["group", "label", "n_new", "promise", "multiplier", "capacity", "capture", "competitor_density", "seed"]
SHORT = {"greedy": "greedy", "cpsat": "cpsat", "cpsat-from-greedy": "cpsat_from_greedy"}
wide = runs[runs.method == "greedy"][KEYS].reset_index(drop=True)
wide["covered_greedy"] = runs[runs.method == "greedy"]["demand_covered"].to_numpy()
wide["seconds_greedy"] = runs[runs.method == "greedy"]["seconds"].to_numpy()
for method in ("cpsat", "cpsat-from-greedy"):
    s = SHORT[method]
    r = runs[runs.method == method].reset_index(drop=True)
    wide[f"covered_{s}"] = r["demand_covered"]
    wide[f"gain_{s}"] = r["demand_covered"] - wide["covered_greedy"]
    wide[f"gain_pct_{s}"] = 100 * wide[f"gain_{s}"] / wide["covered_greedy"]
    wide[f"seconds_{s}"] = r["seconds"]
    wide[f"build_{s}"] = r["build_seconds"]
    wide[f"gap_{s}"] = r["gap"]
    wide[f"status_{s}"] = r["status"]
    wide[f"slow_above_covered_{s}"] = r["slow_above_covered"]
wide["slow_above_covered_greedy"] = runs[runs.method == "greedy"]["slow_above_covered"].to_numpy()
wide["load_before"] = runs.groupby(["group", "label"], sort=False)["load_before"].max().to_numpy()
out = wide.copy()
for col in out.columns:
    if col.startswith(("covered_", "gain_")) and not col.startswith("gain_pct"):
        out[col] = out[col].round(1)
    elif col.startswith("gain_pct"):
        out[col] = out[col].round(2)
    elif col.startswith(("seconds_", "build_")):
        out[col] = out[col].round(2)
    elif col.startswith("gap_"):
        out[col] = out[col].round(4)
    elif col == "load_before":
        out[col] = out[col].round(1)
out.to_csv(FIGS / "04-benchmark.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", None, "display.max_rows", None)
out

## Checks

In [ ]:
def show(name: str, mask: pd.Series) -> None:
    print(f"{name}: {int(mask.sum())}" + (f" -> {wide.loc[mask, 'label'].tolist()}" if mask.any() else ""))

w = wide
show("CP-SAT variant over 10 s", (w.seconds_cpsat > BAR_S) | (w.seconds_cpsat_from_greedy > BAR_S))
show("greedy over 10 s", w.seconds_greedy > BAR_S)
show("cpsat-from-greedy covered < greedy", w.covered_cpsat_from_greedy < w.covered_greedy - EPS)
show("cpsat on its own covered < greedy", w.covered_cpsat < w.covered_greedy - EPS)
show("cpsat on its own covered < cpsat-from-greedy", w.covered_cpsat < w.covered_cpsat_from_greedy - EPS)
show("cpsat on its own not faster than cpsat-from-greedy", w.seconds_cpsat >= w.seconds_cpsat_from_greedy)
for method in planning.METHODS:
    print(method, runs[runs.method == method].status.value_counts().to_dict())
show("any UNKNOWN", (w.status_cpsat == "UNKNOWN") | (w.status_cpsat_from_greedy == "UNKNOWN"))
show("slow_traffic_coverage above covered", w[[c for c in w.columns if c.startswith("slow_above_covered")]].any(axis=1))
cp = runs[runs.method != "greedy"]
bad_gap = cp[cp.gap.isna() | np.isinf(cp.gap.astype(float))]
print(f"CP-SAT rows with a gap None/inf: {len(bad_gap)}" + (f" -> {bad_gap[['label', 'method']].values.tolist()}" if len(bad_gap) else ""))

never_worse = bool((w.covered_cpsat >= w.covered_cpsat_from_greedy - EPS).all())
faster_rows = int((w.seconds_cpsat < w.seconds_cpsat_from_greedy).sum())
median_own, median_hint = w.seconds_cpsat.median(), w.seconds_cpsat_from_greedy.median()
print(f"cpsat on its own never worse on demand covered: {never_worse}")
print(f"median total seconds: cpsat {median_own:.2f}, cpsat-from-greedy {median_hint:.2f}; cpsat faster in {faster_rows} of {len(w)} rows")
served = "cpsat" if never_worse and median_own < median_hint else "cpsat-from-greedy"
print("served variant by the rule:", served)
lo, med, hi = runs.load_before.min(), runs.load_before.median(), runs.load_before.max()
print(f"load before each call: min {lo:.1f}, median {med:.1f}, max {hi:.1f} ({os.cpu_count()} cpus)")

## Figure

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(wide))
for col, name, marker in (("seconds_greedy", "greedy", "o"), ("seconds_cpsat", "cpsat", "s"), ("seconds_cpsat_from_greedy", "cpsat-from-greedy", "^")):
    ax.plot(x, wide[col], marker=marker, linestyle="none", label=name)
ax.axhline(BAR_S, color="k", linestyle="--", linewidth=1, label="10 s")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(wide["label"], rotation=75, ha="right")
ax.set_ylabel("seconds in plan()")
ax.set_title(f"Seconds per plan, by scenario (machine load {lo:.1f}-{hi:.1f} on {os.cpu_count()} cpus)")
ax.legend()
fig.savefig(FIGS / "04-times.png", dpi=120, bbox_inches="tight")
plt.show()

## Combinations off the grid

In [ ]:
COMBINATIONS: list[dict[str, float]] = [   # in range, but off the one-lever grid: each widens the free-flow reach or binds capacity
    {"promise": 16, "multiplier": 1.0},
    {"promise": 15, "multiplier": 1.0},
    {"multiplier": 1.0, "n_new": 20, "capacity": 0.5},
    {"promise": 16, "n_new": 20, "capacity": 0.5},
    {"promise": 16, "capture": 1.5, "competitor_density": 0.0},
]
combo_records: list[dict[str, object]] = []
for kw in COMBINATIONS:
    sc = planning.Scenario(**kw)
    load_before = os.getloadavg()[0]
    p = planning.plan(inputs, sc, "cpsat-from-greedy")
    reach_min = (sc.promise - assumptions.PICK_AND_PACK) / sc.multiplier
    combo_records.append({"scenario": ", ".join(f"{k}={v:g}" for k, v in kw.items()), "free_flow_reach_min": round(reach_min, 1),
                          "status": p.status, "gap": p.gap, "build_seconds": round(p.build_seconds, 2), "seconds": round(p.seconds, 2),
                          "within_10_s": p.seconds <= BAR_S, "gap_stated": p.gap is not None, "load_before": round(load_before, 1)})
combos = pd.DataFrame(combo_records)
combos

## Results (run 2026-11-03, 07:28, headless)

Full area: 4,859 hexes, 1,458 candidate sites, 1,187,469 pairs. 8 cores, OR-Tools 9.15.6755, CP-SAT on 8 workers, solver seed 1,
`PLAN_BUDGET` 9 s. 31 scenarios × 3 methods in 2.4 min, peak memory 0.66 GB. Demand covered is the operator's orders a day; gain is
CP-SAT from greedy over greedy; gap is (bound − objective) / objective. The full table, with households, build times and the
slow-traffic check, is [`figures/04-benchmark.csv`](figures/04-benchmark.csv).

**Load.** The run started after the 1-minute load average had stayed under 4 for 5 minutes (3.8 at the start). It rose to 16.2 by the end:
CP-SAT's own 8 workers add about 8 while it searches, and other work on the machine added the rest in the second half (from the
`capacity` sweep on). The scenarios that hit the time limit return at 9.1-9.2 s either way, because the budget, not the load, ends the
search; the load can still move which of them prove optimality and what gap the others report.

| scenario | greedy | CP-SAT | from greedy | gain (from greedy) | greedy s | CP-SAT s | from greedy s | CP-SAT gap | from greedy gap | CP-SAT status | from greedy status |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- | --- |
| default | 39,360 | 39,360 | 39,360 | 0.00% | 0.04 | 0.29 | 0.24 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| n_new=1 | 28,282 | 28,282 | 28,282 | 0.00% | 0.04 | 2.46 | 2.88 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| n_new=5 | 34,048 | 34,048 | 34,048 | 0.00% | 0.04 | 0.30 | 0.23 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| n_new=15 | 43,351 | 43,380 | 43,380 | 0.07% | 0.04 | 0.27 | 0.33 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| n_new=20 | 46,536 | 46,672 | 46,672 | 0.29% | 0.05 | 0.28 | 0.29 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| promise=8 | 24,945 | 24,981 | 24,981 | 0.14% | 0.03 | 0.22 | 0.22 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| promise=12 | 47,784 | 48,179 | 48,179 | 0.83% | 0.05 | 0.46 | 0.45 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| promise=14 | 52,136 | 52,290 | 52,290 | 0.29% | 0.09 | 1.13 | 1.22 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| promise=16 | 53,861 | 54,405 | 54,405 | 1.01% | 0.12 | 9.22 | 8.79 | 0.34% | 0.00% | FEASIBLE | OPTIMAL |
| multiplier=1 | 53,842 | 54,216 | 54,247 | 0.75% | 0.11 | 9.19 | 8.90 | 0.39% | 0.00% | FEASIBLE | OPTIMAL |
| multiplier=1.4 | 47,784 | 48,179 | 48,179 | 0.83% | 0.05 | 0.47 | 0.51 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| multiplier=2.2 | 30,672 | 30,724 | 30,724 | 0.17% | 0.03 | 0.24 | 0.24 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| multiplier=2.5 | 25,145 | 25,194 | 25,194 | 0.20% | 0.03 | 0.25 | 0.26 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capacity=0.5 | 35,614 | 36,962 | 37,298 | 4.73% | 0.04 | 9.09 | 9.09 | 1.78% | 0.86% | FEASIBLE | FEASIBLE |
| capacity=0.75 | 39,194 | 39,246 | 39,246 | 0.13% | 0.05 | 0.70 | 0.79 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capacity=1.5 | 39,360 | 39,360 | 39,360 | 0.00% | 0.04 | 0.24 | 0.31 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capacity=2 | 39,360 | 39,360 | 39,360 | 0.00% | 0.04 | 0.27 | 0.32 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capture=0.5 | 19,680 | 19,680 | 19,680 | 0.00% | 0.04 | 0.25 | 0.28 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capture=0.75 | 29,520 | 29,520 | 29,520 | 0.00% | 0.04 | 0.27 | 0.28 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capture=1.25 | 49,135 | 49,184 | 49,184 | 0.10% | 0.04 | 0.36 | 0.37 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| capture=1.5 | 58,125 | 58,341 | 58,341 | 0.37% | 0.04 | 9.09 | 9.09 | 0.58% | 0.57% | FEASIBLE | FEASIBLE |
| competitor_density=0 | 44,698 | 44,889 | 44,889 | 0.43% | 0.04 | 0.34 | 0.39 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| competitor_density=0.5 | 42,229 | 42,229 | 42,229 | 0.00% | 0.04 | 0.30 | 0.30 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| competitor_density=1.5 | 37,869 | 37,869 | 37,869 | 0.00% | 0.05 | 0.27 | 0.27 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| competitor_density=2 | 34,802 | 34,802 | 34,802 | 0.00% | 0.04 | 0.23 | 0.34 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| n_new=20, capacity=0.5 | 43,266 | 45,630 | 45,574 | 5.34% | 0.06 | 9.07 | 9.15 | 1.18% | 1.28% | FEASIBLE | FEASIBLE |
| capture=1.5, density=0 | 63,017 | 65,962 | 65,962 | 4.67% | 0.06 | 9.10 | 9.23 | 0.61% | 0.60% | FEASIBLE | FEASIBLE |
| seed=2027 | 39,166 | 39,401 | 39,401 | 0.60% | 0.07 | 0.73 | 1.34 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| seed=2028 | 38,824 | 38,846 | 38,846 | 0.06% | 0.19 | 1.63 | 1.04 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| seed=2029 | 39,121 | 39,386 | 39,386 | 0.68% | 0.35 | 3.39 | 1.51 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |
| seed=2030 | 37,008 | 37,008 | 37,008 | 0.00% | 0.10 | 0.70 | 0.79 | 0.00% | 0.00% | OPTIMAL | OPTIMAL |

![Seconds per plan by scenario](figures/04-times.png)

**What it shows.**
- **Every scenario in the grid returns within 10 s with its gap stated**, for all three methods. The slowest plan took 9.23 s
  (CP-SAT from greedy, capture 1.5× at density 0). Greedy takes 0.03-0.35 s.
- **CP-SAT from greedy is never below greedy** and proves optimality in 27 of 31 scenarios. The 4 it doesn't are the binding-capacity
  and big-market scenarios (capacity 0.5×, the two corners, capture 1.5×): gaps 0.6-1.3%.
- **CP-SAT's gain over greedy is small where capacity is slack** (median 0.14%, 0 in 11 scenarios) **and largest where it binds**:
  4.7% at capacity 0.5×, 5.3% at N 20 with capacity 0.5×, 4.7% at capture 1.5× with no competitor. Greedy's marginal rule picks well
  while every store has room; it can't trade one hex for another when stores fill.
- **CP-SAT on its own** is never below greedy here, but it covers less than CP-SAT from greedy in 2 scenarios (multiplier 1.0 by 31
  orders a day, capacity 0.5× by 336; at promise 16 the check also flags it, by under 0.05 orders a day) and more in one (N 20 at
  capacity 0.5×, by 55). It is faster at the median (0.36 s vs 0.39 s, 22 of 31 scenarios), but the stage 2 rule serves it only if it
  is never worse on demand covered, so **CP-SAT from greedy is the served variant**.
- **Slow-traffic coverage never exceeds demand covered** in any of the 93 plans, FEASIBLE ones included (#33 item 1).
- **Repeat run** (07:36, load 2.4-9.7, the same code): the same checks pass (no plan over 10 s, every gap stated, CP-SAT from greedy
  never below greedy and optimal in 26 of 31, no UNKNOWN, slow-traffic coverage never above demand covered) and the same variant is
  served. CP-SAT on its own was the slower one at the median this time (0.29 s vs 0.27 s): at these times the median is noise.
- **Reproducibility.** CP-SAT runs 8 workers under a time limit, so a FEASIBLE plan can differ run to run (the same seed gives the same
  network, not always the same FEASIBLE plan). OPTIMAL plans repeat their demand covered.

## The promise range: narrowed to 8-16

The first run of this grid swept promise to 20 (the range was 8-20). At promise 20, CP-SAT on its own found no plan (UNKNOWN) and CP-SAT
from greedy kept greedy's plan with no bound, so **no gap could be stated**. Repeated runs of the default scenario at each promise, over three sessions
on 2026-11-02 and -03 (1-minute load 4-16 while they ran; notes give the load where it mattered):

| promise | CP-SAT from greedy, runs | outcome |
| --- | --- | --- |
| 15 | 7 runs | OPTIMAL or FEASIBLE, gap ≤ 8.9%; one run 10.28 s at load 13 |
| 16 | 6 runs | OPTIMAL or FEASIBLE, gap ≤ 0.34%, same bound each run; 8.8-9.3 s, also at load 14 |
| 17 | 5 runs | FEASIBLE, gap 0.2-11.3%, bound moves between runs; up to 9.8 s at load 15 |
| 18 | 6 runs | FEASIBLE; gap 7.8-8.0% in 3 runs, 2,957% in 3 (a bound of 1.65 M orders a day against the area's 89.7 k); one run 10.46 s at load 8 |
| 19, 20 | 2 and 3 runs | CP-SAT on its own UNKNOWN every run; from greedy keeps greedy's plan, **no gap**; 9.2-9.5 s |

Promise 16 is the longest promise whose plan returned in time with a usable gap in every run, so the range is **8-16** (decision log
2026-11-03, cuts.md). The travel-time table still keeps pairs to 17 free-flow minutes; nothing is rebuilt.

**Combinations the grid doesn't sweep.** The grid moves one lever at a time plus two corners (the stage 2 plan), and every row passes.
Combinations inside the narrowed ranges that widen the free-flow reach, (promise − 3) / multiplier, or that bind capacity at a wide
reach, miss. The cell above runs five of them with CP-SAT from greedy (2026-11-03, load 2.8-3.7):

| scenario | free-flow reach | total s | build s | gap |
| --- | ---: | ---: | ---: | --- |
| promise 16, multiplier 1.0 | 13.0 min | 12.21 | 10.30 | none |
| promise 15, multiplier 1.0 | 12.0 min | 10.19 | 8.47 | none |
| multiplier 1.0, N 20, capacity 0.5× | 7.0 min | 9.06 | 2.13 | none |
| promise 16, N 20, capacity 0.5× | 7.2 min | 9.31 | 2.32 | none |
| promise 16, capture 1.5×, density 0 | 7.2 min | 9.48 | 2.28 | none |

The first two run past 10 s mostly on the model build, which the budget doesn't cap (#33 item 3). In all five CP-SAT found no
solution in the budget, so greedy's plan is kept with no bound and no gap (#33 item 4). **The 10 s bar with a gap holds on the grid,
not on every combination in the ranges**; these are known misses for #33, not fixed here. The promise runs above (17-20) and the
promise 18 × multiplier 1.0 run (17.3 s, 14.5 s of build) were measured before the narrowing, with `plan()` in a script, as the range
no longer admits them.